# Stat 220, Unit 6 Homework: Inference Without a Formula

`delivery_routes.csv` is the 500 routes from the Unit 4 homework. `minutes` is how
long a route took. Five columns describe the route: `stops`, `packages`, `miles`, `downtown` and
`rain`. Three more get recorded and have nothing to do with how long anything takes:
`van_age_years`, `dispatcher_rating` and `month`. You will need that last fact.

Run the next two cells first.

In [13]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\Owner\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
PREDICTORS = ["stops", "packages", "miles", "downtown", "rain",
              "van_age_years", "dispatcher_rating", "month"]
routes.head()

,minutes,stops,packages,miles,downtown,rain,van_age_years,dispatcher_rating,month
0,254.3,49,105,55.6,1,0,8,3,7
1,153.5,24,71,32.0,0,0,5,5,2
2,450.9,73,226,55.4,1,1,7,4,4
3,387.1,71,175,73.2,1,0,1,1,12
4,111.2,12,18,32.1,0,1,7,1,11


**Problem 1.** Fit a forest to the routes, and a regression beside it, and find out which
predicts better.

Part a. Report the held-out $R^2$ for both.

In [15]:
X = routes[PREDICTORS]
y = routes["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

forest = RandomForestRegressor(n_estimators=300, random_state=0,
                               n_jobs=-1).fit(X_train, y_train)
line = smf.ols("minutes ~ " + " + ".join(PREDICTORS),
               data=routes.loc[X_train.index]).fit()
pred = line.predict(routes.loc[X_test.index])
line_r2 = 1 - ((y_test - pred) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()

print("forest     held-out R2:", round(forest.score(X_test, y_test), 3))
print("regression held-out R2:", round(line_r2, 3))

forest     held-out R2: 0.955
regression held-out R2: 0.963


The R2 for the forest is .955 and the R2 for the regression model is .963



Part b. One of the two can hand you a coefficient with an interval and the other cannot. Say which, and say what you would have to do to get something comparable out of the one that cannot.

The regression model can provide coefficients with confidence intervals because it estimates the relationship between each predictor and the response variable. To get something comparable, we would need to use bootstrapping by repeatedly fitting the forest on resampled training data and examining the distribution of its results.



**Problem 2.** Now ask which columns it is using. Run both importance methods and compare them.

Part a. Report both rankings in one table.

In [16]:
impurity = pd.Series(forest.feature_importances_, index=PREDICTORS)
perm = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=0, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=PREDICTORS)

print(pd.DataFrame({"impurity": impurity.round(4),
                    "permutation": permutation.round(4)}).sort_values("impurity",
                                                                      ascending=False))

                   impurity  permutation
stops                0.9127       1.1920
packages             0.0416       0.0326
miles                0.0312       0.0463
month                0.0049       0.0011
van_age_years        0.0043       0.0003
dispatcher_rating    0.0032       0.0003
downtown             0.0010       0.0000
rain                 0.0010       0.0006


                    impurity  permutation
stops                0.9127       1.1920

packages             0.0416       0.0326

miles                0.0312       0.0463

month                0.0049       0.0011

van_age_years        0.0043       0.0003

dispatcher_rating    0.0032       0.0003

downtown             0.0010       0.0000

rain                 0.0010       0.0006





Part b. Both methods put `month` above `downtown` and `rain`. Two of those three columns are real and one is not. Say which, and explain in two or three sentences how a real predictor can finish below a column that means nothing.

month is the predictor that does not actually affect route duration. downtown and rain are real predictors because they describe conditions that can affect how long deliveries take. A real predictor can rank below a meaningless column because random forests can sometimes find accidental patterns in unrelated data



**Problem 3.** A ranking is not the same as an effect. Fit a regression on the same eight columns and see what it says instead.

Part a. Report the coefficient and p-value for `downtown`, `rain` and `month`.

In [17]:
fit = smf.ols("minutes ~ " + " + ".join(PREDICTORS), data=routes).fit()
for c in ["downtown", "rain", "month"]:
    print(f"{c:<10} coef {fit.params[c]:+7.3f}   p {fit.pvalues[c]:.2g}")

downtown   coef  +4.144   p 0.053
rain       coef  +6.699   p 0.0079
month      coef  +0.014   p 0.96


downtowns coefficient is 4.144 and its p-value is 0.053
rains coefficient is 6.699 and its p-value is 0.0079
months coefficient is 0.014 and its p-value is 0.96



Part b. The forest ranked `month` above `rain`. The regression says something different. Explain in three or four sentences what each of the two is measuring, and which one answers the question 'does rain add time to a route'.

The random forest measures how useful a variable is for making accurate predictions, while regression measures the relationship between a predictor and delivery time while holding the other predictors constant. This is why the forest can rank month higher than rain, even though month has no actual relationship with delivery time. The regression shows that rain adds approximately seven minutes to a delivery route when the other variables are held constant.



**Problem 4.** A ranking does not tell you whether a column is doing anything at all, or
whether it just happened to land above one that is doing nothing. Resampling does. Resample the
training rows only, and score against the test set you already have.

Part a. Report the point estimate and a 95% bootstrap interval for each column, and say which ones you cannot separate from zero.

In [18]:
rng = np.random.default_rng(6)

def importance_of(X_fit, y_fit, seed):
    f = RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1).fit(X_fit, y_fit)
    r = permutation_importance(f, X_test, y_test, n_repeats=4, random_state=seed, n_jobs=-1)
    return pd.Series(r.importances_mean, index=PREDICTORS)

base = importance_of(X_train, y_train, 0)
reps = pd.DataFrame([importance_of(X_train.iloc[take], y_train.iloc[take], k)
                     for k, take in enumerate(rng.integers(0, len(X_train),
                                                           (30, len(X_train))))])

for c in PREDICTORS:
    lo, hi = np.percentile(reps[c], [2.5, 97.5])
    flag = "  cannot separate from zero" if lo <= 0 <= hi else ""
    print(f"{c:<18} {base[c]:+.4f}  [{lo:+.4f}, {hi:+.4f}]{flag}")

stops              +1.2278  [+1.0232, +1.3415]
packages           +0.0273  [+0.0146, +0.0568]
miles              +0.0458  [+0.0304, +0.0588]
downtown           +0.0001  [-0.0009, +0.0010]  cannot separate from zero
rain               +0.0008  [-0.0007, +0.0006]  cannot separate from zero
van_age_years      +0.0006  [-0.0005, +0.0020]  cannot separate from zero
dispatcher_rating  +0.0004  [-0.0025, +0.0012]  cannot separate from zero
month              +0.0006  [-0.0001, +0.0028]  cannot separate from zero


                    point       interval   
stops              +1.2278  [+1.0232, +1.3415]

packages           +0.0273  [+0.0146, +0.0568]

miles              +0.0458  [+0.0304, +0.0588]

downtown           +0.0001  [-0.0009, +0.0010]  cannot separate from zero

rain               +0.0008  [-0.0007, +0.0006]  cannot separate from zero

van_age_years      +0.0006  [-0.0005, +0.0020]  cannot separate from zero

dispatcher_rating  +0.0004  [-0.0025, +0.0012]  cannot separate from zero

month              +0.0006  [-0.0001, +0.0028]  cannot separate from zero




Part b. In problem 3 the regression found `rain` real, with about seven extra minutes and a small p-value. Here the importance of `rain` cannot be separated from zero. Both results are correct. Explain in three or four sentences how that happens, and say which of the two you would report to someone asking whether rain slows deliveries.

Regression measures the relationship between rain and delivery time while controlling for the other predictors, whereas permutation importance measures how much the forest's predictions get worse when rain is shuffled. If other variables already explain much of the information associated with rain, shuffling rain might not noticeably hurt the forest's predictions. I would report the regression result when answering whether rain adds time to deliveries because it directly estimates the additional minutes associated with rain.


**Problem 5.** No computer for this one. Three colleagues, three claims.

Part a. A colleague fits a forest to customer data and reports that `account_number` is the second most important variable, using the default importance. Write the two or three sentences you would say in response.

I would question why account_number is ranked so highly because an account number should not have a meaningful relationship with customer behavior. Random forests can sometimes assign high importance to meaningless variables because they find accidental patterns in the data.



Part b. Another colleague bootstraps a sample of 40 patients, gets a 95% interval, and says it shows the treatment works in the wider population. The 40 patients were all recruited from one clinic. Explain what the interval does and does not cover.

The confidence interval describes the uncertainty in the estimated treatment effect for the sample of patients recruited from that clinic. However, it does not account for whether those 40 patients represent the wider population. Since all the patients came from one clinic, the results might not apply to people from different clinics or backgrounds.



Part c. A third colleague resamples the whole dataset, then splits each resample into training and test rows, and reports bootstrap intervals for permutation importance. Every column comes out important. Say what went wrong.

The problem is that they resampled the entire dataset before splitting it into training and test sets. This can cause data leakage because some of the same original observations can appear in both the training and test sets. As a result, the test set is no longer independent, and the importance estimates can be misleading.

